## Init

In [2]:
# %pip install numpy==1.26.4, scipy==1.13.1
# %pip install -U ipywidgets
# %pip install -U gradio
# %pip install txagent
# %pip install tooluniverse

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### Tool Universe

In [4]:
from tooluniverse import ToolUniverse

tu = ToolUniverse()
tu.load_tools()

ℹ️  Number of tools after load tools: 783
ℹ️  Found 4 MCP Auto Loader tool(s), processing...
MCPAutoLoaderTool 'mcp_auto_loader_txagent' initialized with:
  - server_url: http://your_api_key_here:7000/mcp
  - transport: http
  - auto_register: True
  - tool_prefix: mcp_
  - selected_tools: None
  - timeout: 5
⚠️  Warning: Cannot process MCP Auto Loader 'mcp_auto_loader_txagent' because we're already in an event loop.
MCPAutoLoaderTool 'mcp_auto_loader_expert_feedback' initialized with:
  - server_url: http://your_api_key_here/mcp
  - transport: http
  - auto_register: True
  - tool_prefix: expert_
  - selected_tools: None
  - timeout: 5
⚠️  Warning: Cannot process MCP Auto Loader 'mcp_auto_loader_expert_feedback' because we're already in an event loop.
MCPAutoLoaderTool 'mcp_auto_loader_uspto_downloader' initialized with:
  - server_url: http://your_api_key_here:8081/mcp
  - transport: http
  - auto_register: True
  - tool_prefix: mcp_
  - selected_tools: None
  - timeout: 5
⚠️  Warnin

In [5]:
tools = tu.run({
    "name": "Tool_Finder_Keyword",
    "arguments": {"description": "disease target associations", "limit": 10}
})

In [6]:
tools = tu.run({"name":"FDA_get_drug_names_by_clinical_studies", "arguments":{"clinical_studies":"deep vein thrombosis", "indication":"treatment", "limit":5}})

https://api.fda.gov/drug/label.json?limit=5&search=(clinical_studies:(deep+AND+vein+AND+thrombosis)+indications_and_usage:(treatment))+AND+(_exists_:openfda.brand_name+_exists_:openfda.generic_name)+AND+(_exists_:clinical_studies+_exists_:indications_and_usage)


In [7]:
tools

{'meta': {'skip': 0, 'limit': 5, 'total': 28776},
 'results': [{'clinical_studies': None,
   'indications_and_usage': 'Uses for the treatment of acne',
   'openfda.brand_name': ['Quick Action'],
   'openfda.generic_name': ['SALICYLIC ACID']},
  {'clinical_studies': 'Kaplan-Meier Curves for Investigator-Assessed Relapse-Free Survival in COMBI-AD in the Adjuvant Treatment of Melanoma Figure 4.......In the METRIC study, patients were not permitted to have more than one prior chemotherapy regimen for advanced or metastatic disease; prior treatment with a BRAF inhibitor or MEK inhibitor was not permitted.......Treatment continued until disease progression or unacceptable toxicity.......The median durations of follow-up prior to initiation of alternative treatment were 4.9 months for patients treated with MEKINIST and 3.1 months for patients treated with chemotherapy.......The COMBI-d study compared dabrafenib plus MEKINIST to dabrafenib plus placebo as first-line treatment for patients with

In [8]:
# Get all tool names from ToolUniverse
if isinstance(tu.all_tools, dict):
    available_tools = set(tu.all_tools.keys())
elif isinstance(tu.all_tools, list):
    # Extract tool names from list of dicts
    available_tools = set(t.get('name', t) if isinstance(t, dict) else t for t in tu.all_tools)
else:
    available_tools = set()
    
print(f"Total tools in ToolUniverse: {len(available_tools)}")
print(f"\nSample tools: {list(available_tools)[:10]}")

# Identify meta/agent tools to remove (tools that delegate to other systems)
META_AGENT_TOOLS = {
    # Agent/delegation tools
    'CallAgent',
    'DiseaseAnalyzerAgent', 
    'MedicalLiteratureReviewer',
    'LiteratureSynthesisAgent',
    'ClinicalTrialDesignAgent',
    'DrugSafetyAnalyzer',
    'DrugInteractionAnalyzerAgent',
    'BiomarkerDiscoveryWorkflow',
    
    # Tool finder/discovery tools (model should select directly)
    'Tool_Finder',
    'Tool_Finder_Keyword', 
    'Tool_Finder_LLM',
    'ToolDiscover',
    'Tool_RAG',
    
    # Other meta tools
    'Finish',
}
ALL_TOOLS = set(available_tools) - META_AGENT_TOOLS

print(f"FINAL {len(ALL_TOOLS)} TOOLS: {ALL_TOOLS}")

Total tools in ToolUniverse: 783

Sample tools: ['FDA_get_manufacturer_name_NDC_number_by_drug_name', 'cancer_gene_census_disease_target_score', 'get_clinical_trial_references', 'euhealthinfo_search_infectious_diseases', 'get_scikit_learn_info', 'ToolGraphGenerationPipeline', 'FDA_get_drug_names_by_active_ingredient', 'embedding_sync_download', 'OpenTargets_get_similar_entities_by_disease_efoId', 'OpenTargets_get_associated_drugs_by_target_ensemblID']
FINAL 769 TOOLS: {'FDA_get_manufacturer_name_NDC_number_by_drug_name', 'cancer_gene_census_disease_target_score', 'get_clinical_trial_references', 'euhealthinfo_search_infectious_diseases', 'get_scikit_learn_info', 'ToolGraphGenerationPipeline', 'FDA_get_drug_names_by_active_ingredient', 'embedding_sync_download', 'OpenTargets_get_similar_entities_by_disease_efoId', 'OpenTargets_get_associated_drugs_by_target_ensemblID', 'mesh_get_subjects_by_subject_scope_or_definition', 'FDA_get_drug_names_by_overdosage_info', 'get_umap_learn_info', 'FD

In [9]:
spec = tu.tool_specification("CallAgent")
print(spec)

spec = tu.tool_specification("Tool_Finder_Keyword")
print(spec)

spec = tu.tool_specification("Tool_Finder")
print(spec)

{'type': 'SpecialTool', 'name': 'CallAgent', 'description': 'Give a solution plan to the agent and let it solve the problem. Solution plan should reflect a distinct method, approach, or viewpoint to solve the given question. Call these function multiple times, and each solution plan should start with different aspects of the question, for example, genes, phenotypes, diseases, or drugs, etc. The CallAgent will achieve the task based on the plan, so only give the plan instead of unverified information.', 'parameter': {'type': 'object', 'properties': {'solution': {'type': 'string', 'description': 'A feasible and concise solution plan that address the question.', 'required': True}}, 'required': ['solution']}}
{'type': 'ToolFinderKeyword', 'name': 'Tool_Finder_Keyword', 'description': 'Simple keyword-based tool finder for discovering relevant tools using text matching', 'parameter': {'type': 'object', 'properties': {'description': {'type': 'string', 'description': 'The description of the to

In [10]:
specs = tu.get_tool_specification_by_names([
    "FAERS_count_reactions_by_drug_event",
    "OpenTargets_get_associated_targets_by_disease_efoId"
])
print(specs)

[{'type': 'FDADrugAdverseEventTool', 'name': 'FAERS_count_reactions_by_drug_event', 'description': 'Count the number of adverse reactions reported for a given drug. Only medicinalproduct is required; all other filters (patientsex, patientagegroup, occurcountry, serious, seriousnessdeath, reactionmeddraverse) are optional. When reactionmeddraverse is not specified, returns all adverse reactions (AE) with their counts grouped by MedDRA Preferred Term. When reactionmeddraverse is specified, filters results to only include that specific MedDRA Lowest Level Term. Use filters sparingly to avoid overly restrictive searches that return no results. Data source: FDA Adverse Event Reporting System (FAERS).', 'parameter': {'type': 'object', 'properties': {'medicinalproduct': {'type': 'string', 'description': 'Drug name.', 'required': True}, 'patientsex': {'type': 'string', 'enum': ['Male', 'Female'], 'description': "Optional: Filter by patient sex. Omit this parameter if you don't want to filter b

In [11]:
print(f"✅ Loaded {len(tu.all_tools)} scientific tools!")

for tool in tu.all_tools:
    print(tool)


✅ Loaded 783 scientific tools!
{'type': 'SpecialTool', 'name': 'Finish', 'description': 'Indicate the end of multi-step reasoning.', 'parameter': {'type': 'object', 'properties': {}, 'required': []}}
{'type': 'SpecialTool', 'name': 'CallAgent', 'description': 'Give a solution plan to the agent and let it solve the problem. Solution plan should reflect a distinct method, approach, or viewpoint to solve the given question. Call these function multiple times, and each solution plan should start with different aspects of the question, for example, genes, phenotypes, diseases, or drugs, etc. The CallAgent will achieve the task based on the plan, so only give the plan instead of unverified information.', 'parameter': {'type': 'object', 'properties': {'solution': {'type': 'string', 'description': 'A feasible and concise solution plan that address the question.'}}, 'required': ['solution']}}
{'type': 'ToolFinderEmbedding', 'name': 'Tool_RAG', 'description': 'Retrieve related tools from the too

## Preprocess - Reasoning Data

In [12]:
import json
import glob
from collections import Counter

# Load all tool selection data
all_reason_data = []
files = glob.glob('datasets/finetuning/model2_reasoning*.jsonl')

print("=== Dataset Files ===")
for f in sorted(files):
    with open(f, 'r') as file:
        samples = [json.loads(line) for line in file]
        print(f"{f}: {len(samples)} samples")
        all_reason_data.extend(samples)

print(f"\n=== Total: {len(all_reason_data)} samples ===")

=== Dataset Files ===
datasets/finetuning/model2_reasoning.jsonl: 465 samples
datasets/finetuning/model2_reasoning_gpu0.jsonl: 362 samples
datasets/finetuning/model2_reasoning_gpu2.jsonl: 71 samples
datasets/finetuning/model2_reasoning_gpu3.jsonl: 737 samples
datasets/finetuning/model2_reasoning_gpu4.jsonl: 154 samples
datasets/finetuning/model2_reasoning_gpu5.jsonl: 1106 samples
datasets/finetuning/model2_reasoning_gpu6.jsonl: 277 samples
datasets/finetuning/model2_reasoning_gpu7.jsonl: 1139 samples

=== Total: 4311 samples ===


In [13]:
import re
import json
import sys

def extract_tool_calls(text: str, update_limit_to_1 = False):
    """
    Extract tool call info from text of the form:
      [ToolName(arg1=val1, arg2=val2)]

    Returns a list of dicts:
      [{"name": "ToolName", "arguments": {...}}, ...]
    """
    # Match things like [EuropePMC_Guidelines_Search(query=..., limit=5)]
    pattern = r'\[([A-Za-z0-9_]+)\((.*?)\)\]'
    matches = re.findall(pattern, text, flags=re.DOTALL)

    tool_calls = []

    for tool_name, args_str in matches:
        args = {}

        # Skip if there are no arguments
        if args_str.strip():
            # Split on commas between arguments
            # (safe here because your example values don't contain commas)
            for part in args_str.split(','):
                part = part.strip()
                if not part:
                    continue
                if '=' in part:
                    key, value = part.split('=', 1)
                    key = key.strip()
                    value = value.strip()
                    args[key] = value

        args["limit"] = 1
        
        if tool_name in ALL_TOOLS:
            tool_calls.append({
                "name": tool_name,
                "arguments": args
            })

    return tool_calls

# def regenerate_tool_response(tools):
#     res = []
#     for tool in tools:
#         new_tool_res = json.dumps(tu.run(tool))
#         if "Invalid Query" not in new_tool_res:
#             res.append(new_tool_res)
#     return res
        

In [14]:
import pandas as pd
import json

def save_predata_jsonl(pre_data, path):
    """
    Save a list of dictionaries (pre_data) to a JSONL file using pandas.
    """
    # Convert list of dicts → DataFrame
    df = pd.DataFrame(pre_data)

    # Write JSON Lines
    df.to_json(
        path,
        orient="records",
        lines=True
    )

    print(f"Wrote {len(df)} records to {path}")

# ---- Example usage ----

# pre_data = regenerate_all_tool_responses_parallel(all_reason_data)

output_path = "datasets/finetuning/model2_processed_reasoning_with_regenerate_tools.jsonl"

In [15]:
import json
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


def extract_question_and_tools(dp_input: str):
    """Split the raw input into question text and tools part."""
    question_part, tools_part = dp_input.split("TOOL RESULTS", 1)
    # Adjust this split if your question format changes
    question = question_part.split('QUESTION:\nQuestion: ')[1].strip()
    return question, tools_part


def regenerate_all_tool_responses_parallel(all_reason_data, output_path, max_workers=32):
    """
    Parallelize tu.run across ALL tools from ALL datapoints.
    After every completed tool call, recompute pre_data (with current
    tool outputs) and save to file.
    """
    # 1) Parse all datapoints to extract questions + tool calls
    parsed = []          # index aligned with all_reason_data
    jobs = []            # (dp_index, tool) tuples

    for idx, dp in enumerate(all_reason_data):
        question, tools_raw = extract_question_and_tools(dp["input"])
        tool_calls = extract_tool_calls(tools_raw)  # your existing function

        parsed.append({
            "dp": dp,
            "question": question,
            "tools": tool_calls,
        })

        for tool in tool_calls:
            jobs.append((idx, tool))

    # 2) Run all tu.run(tool) in parallel
    tool_outputs = defaultdict(list)  # dp_index -> list of tool result strings

    def run_single_tool(dp_index, tool):
        """Wrapper so we can return which datapoint this result belongs to."""
        result_json_str = json.dumps(tu.run(tool))
        if "NOT_FOUND" in result_json_str:
            return dp_index, None
        return dp_index, result_json_str

    def build_pre_data(parsed, tool_outputs):
        """Build pre_data from current tool_outputs snapshot."""
        pre_data = []
        for idx, item in enumerate(parsed):
            dp = item["dp"]
            question = item["question"]
            new_tool_res = tool_outputs.get(idx, [])

            reasoning = dp["output"]
            answer = dp["ground_truth_answer"]

            if answer in reasoning:
                pre_data.append({
                    **dp,
                    "input": (
                        "## TOOL RESULT\n"
                        f"{json.dumps(new_tool_res)}\n\n"
                        "## QUESTION\n"
                        f"{question}"
                    ),
                    "with_tool_info": answer in json.dumps(new_tool_res),
                    "metadata": {
                        "question": question,
                        "regenerated_tool_responses": new_tool_res,
                    },
                })
        return pre_data

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_job = {
            executor.submit(run_single_tool, dp_index, tool): (dp_index, tool)
            for dp_index, tool in jobs
        }

        for future in as_completed(future_to_job):
            dp_index, result = future.result()
            if result is not None:
                tool_outputs[dp_index].append(result)

            # After every completed tool call, rebuild and save snapshot
            pre_data_snapshot = build_pre_data(parsed, tool_outputs)
            save_predata_jsonl(pre_data_snapshot, output_path)

    # Final full build & return
    final_pre_data = build_pre_data(parsed, tool_outputs)
    return final_pre_data


In [16]:
# dp0 = all_reason_data[0]
# current = dp0["input"].split("TOOL RESULTS")[1]
# new = extract_tool_calls(current)
# new_tool_res = regenerate_tool_response(new)

In [17]:
dp0 = all_reason_data[0]
dp0

{'instruction': 'Based on the medical question and tool results, select the best answer option and explain your reasoning.',
 'input': 'QUESTION:\nQuestion: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?\nOptions: Ampicillin / Ceftriaxone / Ciprofloxacin / Doxycycline / Nitrofurantoin\n\nTOOL RESULTS:\n[FDA_get_indications_by_drug_name(drug_name=Ampicillin, limit=5)]\n{\'meta\': {\'skip\': 0, \'limit\': 5, \'total\': 74}, \'results\': [{\'openfda.brand_name\'

In [18]:
import os
import json
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed


def extract_question_and_tools(dp_input: str):
    """Split the raw input into question text and tools part."""
    question_part, tools_part = dp_input.split("TOOL RESULTS", 1)
    # Adjust this split if your question format changes
    question = question_part.split('QUESTION:\nQuestion: ')[1].strip()
    return question, tools_part


def regenerate_all_tool_responses_parallel(
    all_reason_data,
    output_path,
    max_workers=32,
):
    """
    Parallelize tu.run across ALL tools from ALL datapoints.

    - Uses `output_path` as a checkpoint file (JSONL).
    - If a datapoint's processed record is already present in the file
      (identified by `_dp_index`), it will NOT be recomputed.
    - As soon as all tools for a datapoint are done, we append one JSONL
      line for that datapoint and free its in-memory state.

    Returns:
        int: number of NEW datapoints written in this run.
    """

    # --- 0) Read already processed datapoints from existing file (if any) ---
    processed_dp_indices = set()
    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    # skip malformed lines
                    continue
                idx = rec.get("_dp_index")
                if idx is not None:
                    processed_dp_indices.add(idx)

    # --- 1) Parse only UNPROCESSED datapoints: questions + tool calls ---
    parsed = {}          # dp_index -> {"dp": dp, "question": question}
    jobs = []            # list of (dp_index, tool)
    tool_counts = {}     # dp_index -> total # of tools for that dp
    answers = {}         # dp_index -> ground_truth_answer
    reasonings = {}      # dp_index -> reasoning/output text

    for idx, dp in enumerate(all_reason_data):
        # Skip if already processed in a previous run
        if idx in processed_dp_indices:
            continue

        question, tools_raw = extract_question_and_tools(dp["input"])
        tool_calls = extract_tool_calls(tools_raw)  # your existing function

        parsed[idx] = {
            "dp": dp,
            "question": question,
        }
        answers[idx] = dp["ground_truth_answer"]
        reasonings[idx] = dp["output"]

        tool_counts[idx] = len(tool_calls)

        for tool in tool_calls:
            jobs.append((idx, tool))

    # If there's nothing new to process, just return.
    if not parsed:
        print("No new datapoints to process; everything is already in the JSONL file.")
        return 0

    # --- 2) State during execution (for UNPROCESSED datapoints only) ---
    tool_outputs = defaultdict(list)   # dp_index -> list of tool result JSON strings
    done_counts = defaultdict(int)     # dp_index -> # of tools finished
    dp_written = set()                 # which dp indices are already written this run
    answer_seen_in_tools = defaultdict(bool)  # dp_index -> bool

    new_written_count = 0  # number of new records written in THIS run

    def run_single_tool(dp_index, tool):
        """
        Wrapper so we can return which datapoint this result belongs to.
        Protect against exceptions so one bad tool doesn't kill everything.
        """
        try:
            res = tu.run(tool)
            result_json_str = json.dumps(res)
        except Exception:
            # You can log/print the exception if you want
            return dp_index, None

        if "NOT_FOUND" in result_json_str:
            return dp_index, None
        return dp_index, result_json_str

    def maybe_finalize_dp(dp_index, file_obj):
        """
        If all tools for a datapoint are done and we haven't written it yet,
        build its record and append one JSONL line to file.
        Then free memory for that datapoint.
        """
        nonlocal new_written_count

        if dp_index in dp_written:
            return

        # Not all tools for this dp are done yet
        if done_counts[dp_index] < tool_counts.get(dp_index, 0):
            return

        item = parsed.get(dp_index)
        if item is None:
            dp_written.add(dp_index)
            return

        dp = item["dp"]
        question = item["question"]
        new_tool_res = tool_outputs.get(dp_index, [])

        reasoning = reasonings[dp_index]
        answer = answers[dp_index]

        # Only keep datapoints where answer appears in reasoning
        if answer in reasoning:
            record = {
                **dp,
                # keep a stable index so we can skip this dp next time
                "_dp_index": dp_index,
                "input": (
                    "## TOOL RESULT\n"
                    f"{json.dumps(new_tool_res)}\n\n"
                    "## QUESTION\n"
                    f"{question}"
                ),
                # use the incremental flag, avoids scanning giant string
                "with_tool_info": bool(answer_seen_in_tools[dp_index]),
                "metadata": {
                    "question": question,
                    "regenerated_tool_responses": new_tool_res,
                },
            }

            # Append one JSONL line (streaming, no pandas; better for huge rows)
            file_obj.write(json.dumps(record, ensure_ascii=False) + "\n")
            file_obj.flush()
            new_written_count += 1

        # Mark as written and free memory for this dp
        dp_written.add(dp_index)
        tool_outputs.pop(dp_index, None)
        done_counts.pop(dp_index, None)
        tool_counts.pop(dp_index, None)
        answers.pop(dp_index, None)
        reasonings.pop(dp_index, None)
        parsed.pop(dp_index, None)
        answer_seen_in_tools.pop(dp_index, None)

    # --- 3) Handle datapoints with ZERO tools upfront ---
    # (otherwise they never get finalized because no futures will complete for them)
    with open(output_path, "a", encoding="utf-8") as f:  # append mode for checkpointing
        for idx in list(parsed.keys()):
            if tool_counts.get(idx, 0) == 0:
                done_counts[idx] = 0
                maybe_finalize_dp(idx, f)

        # --- 4) Run all tool calls in parallel and stream results to disk ---
        if jobs:
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_to_job = {
                    executor.submit(run_single_tool, dp_index, tool): (dp_index, tool)
                    for dp_index, tool in jobs
                    if dp_index in parsed  # ensure not already finalized (zero-tools)
                }

                for future in as_completed(future_to_job):
                    dp_index, result = future.result()

                    # If this dp was already finalized (e.g., zero-tools), skip.
                    if dp_index not in parsed:
                        continue

                    done_counts[dp_index] += 1

                    if result is not None:
                        tool_outputs[dp_index].append(result)
                        # update answer_in_tools incrementally
                        if (not answer_seen_in_tools[dp_index] and
                                answers[dp_index] in result):
                            answer_seen_in_tools[dp_index] = True

                    # If this datapoint is now complete, write its record & free its memory
                    maybe_finalize_dp(dp_index, f)

    return new_written_count


In [19]:
# pre_data = []
# for dp in all_reason_data:
#     question, tools = dp["input"].split("TOOL RESULTS")[0], dp["input"].split("TOOL RESULTS")[1]
    
#     new = extract_tool_calls(tools)
#     new_tool_res = regenerate_tool_response(new)
    
#     reasoning = dp["output"]
#     answer = dp["ground_truth_answer"]
    
#     if answer in reasoning:
#         pre_data.append({
#             **dp,
#             "input": f"## TOOL RESULT\n{json.dumps(new_tool_res)}\n\n## QUESTION\n{question.split('QUESTION:\nQuestion: ')[1].strip()}",
#             "with_tool_info": answer in json.dumps(new_tool_res),
#             "metadata": {
#                 "question": question.split('QUESTION:\nQuestion: ')[1].strip(),
#                 "regenerated_tool_responses": new_tool_res
#             }
#         })
    
pre_data = regenerate_all_tool_responses_parallel(all_reason_data, max_workers=16, output_path=output_path)

https://api.fda.gov/drug/label.json?limit=1&search=(openfda.brand_name:(Ceftriaxone)+openfda.generic_name:(CEFTRIAXONE))+AND+(_exists_:indications_and_usage)+AND+(_exists_:openfda.brand_name+_exists_:openfda.generic_name)https://api.fda.gov/drug/label.json?limit=1&search=(openfda.brand_name:(Ciprofloxacin)+openfda.generic_name:(CIPROFLOXACIN))+AND+(_exists_:indications_and_usage)+AND+(_exists_:openfda.brand_name+_exists_:openfda.generic_name)
https://api.fda.gov/drug/label.json?limit=1&search=(openfda.brand_name:(Doxycycline)+openfda.generic_name:(DOXYCYCLINE))+AND+(_exists_:indications_and_usage)+AND+(_exists_:openfda.brand_name+_exists_:openfda.generic_name)

https://api.fda.gov/drug/label.json?limit=1&search=(openfda.brand_name:(Ampicillin)+openfda.generic_name:(AMPICILLIN))+AND+(_exists_:indications_and_usage)+AND+(_exists_:openfda.brand_name+_exists_:openfda.generic_name)
https://api.fda.gov/drug/label.json?limit=1&search=(openfda.brand_name:(Ampicillin)+openfda.generic_name:(AMPI

Tool GWASGeneSearch not found in module tooluniverse.gwas_tool after import
Tool GWASGeneSearch not found in lazy registry, falling back to full discovery
Could not import blast_tool: No module named 'Bio'



🔗 Request URL: https://medlineplus.gov/download/genetics/condition/Neural tube.json

🔗 Request URL: https://medlineplus.gov/download/genetics/condition/Surface ectoderm.json
Invalid Query:  {'code': 'NOT_FOUND', 'message': 'No matches found!'}

🔗 Request URL: https://medlineplus.gov/download/genetics/condition/Neural crest.json

🔗 Request URL: https://medlineplus.gov/download/genetics/condition/Notochord.json
Invalid Query:  {'code': 'NOT_FOUND', 'message': 'No matches found!'}

🔗 Request URL: https://medlineplus.gov/download/genetics/condition/Mesoderm.json
Invalid Query:  {'code': 'NOT_FOUND', 'message': 'No matches found!'}

📊 Response status: 200
📏 Response length: 22124 characters
🔤 First 500 characters of response: <?xml version="1.0" encoding="UTF-8"?>
<nlmSearchResult>
  <term>neuromuscular disorder</term>
  <file>viv_6LqOOd</file>
  <server>pvlb7srch15</server>
  <count>6</count>
  <retstart>0</retstart>
  <retmax>10</retmax>
  <list num="6" start="0" per="10">
    <document 

: 

In [ ]:
len(pre_data)

In [ ]:
import pandas as pd
import json

def save_predata_jsonl(pre_data, path):
    """
    Save a list of dictionaries (pre_data) to a JSONL file using pandas.
    """
    # Convert list of dicts → DataFrame
    df = pd.DataFrame(pre_data)

    # Write JSON Lines
    df.to_json(
        path,
        orient="records",
        lines=True
    )

    print(f"Wrote {len(df)} records to {path}")

# ---- Example usage ----

# pre_data = regenerate_all_tool_responses_parallel(all_reason_data)

output_path = "datasets/finetuning/model2_reasoning_processed_no_regenerate_tools.jsonl"
save_predata_jsonl(pre_data, output_path)

In [ ]:
SYSTEM_PROMPT_REASON = """You are a medical reasoning assistant. Given a medical question and the corresponding tool results (e.g., guidelines, studies, drug information), select the single best answer option and provide a concise explanation that clearly links your reasoning to the tool results."""

## Preprocess - Tool Selection Data

In [ ]:
import json
import glob
from collections import Counter

# Load all tool selection data
all_data = []
files = glob.glob('datasets/finetuning/model1_tool_selection*.jsonl')

print("=== Dataset Files ===")
for f in sorted(files):
    with open(f, 'r') as file:
        samples = [json.loads(line) for line in file]
        print(f"{f}: {len(samples)} samples")
        all_data.extend(samples)

print(f"\n=== Total: {len(all_data)} samples ===")

=== Dataset Files ===
datasets/finetuning/model1_tool_selection.jsonl: 464 samples
datasets/finetuning/model1_tool_selection_gpu0.jsonl: 362 samples
datasets/finetuning/model1_tool_selection_gpu2.jsonl: 71 samples
datasets/finetuning/model1_tool_selection_gpu3.jsonl: 737 samples
datasets/finetuning/model1_tool_selection_gpu4.jsonl: 154 samples
datasets/finetuning/model1_tool_selection_gpu5.jsonl: 1106 samples
datasets/finetuning/model1_tool_selection_gpu6.jsonl: 277 samples
datasets/finetuning/model1_tool_selection_gpu7.jsonl: 1139 samples
datasets/finetuning/model1_tool_selection_missing_gpu0.jsonl: 6 samples
datasets/finetuning/model1_tool_selection_missing_gpu1.jsonl: 436 samples
datasets/finetuning/model1_tool_selection_missing_gpu2.jsonl: 367 samples
datasets/finetuning/model1_tool_selection_missing_gpu3.jsonl: 836 samples
datasets/finetuning/model1_tool_selection_missing_gpu4.jsonl: 51 samples
datasets/finetuning/model1_tool_selection_missing_gpu5.jsonl: 898 samples
datasets/fine

In [4]:
# Analyze tool usage
all_tools = []
tool_counts = Counter()

for sample in all_data:
    output = sample.get('output', '')
    # Parse tool names from output (format: "tool_name: {args}" or "tool_name: value")
    for line in output.split('\n'):
        if ':' in line:
            tool_name = line.split(':')[0].strip()
            if tool_name:
                all_tools.append(tool_name)
                tool_counts[tool_name] += 1

print(f"=== Tool Statistics ===")
print(f"Total tool calls: {len(all_tools)}")
print(f"Unique tools: {len(tool_counts)}")
print(f"\n=== Top 50 Most Used Tools ===")
for tool, count in tool_counts.most_common(50):
    print(f"  {tool}: {count}")

=== Tool Statistics ===
Total tool calls: 74305
Unique tools: 614

=== Top 50 Most Used Tools ===
  CallAgent: 8907
  DiseaseAnalyzerAgent: 2918
  Tool_Finder_Keyword: 2800
  Tool_Finder: 2641
  get_HPO_ID_by_phenotype: 2359
  MedicalLiteratureReviewer: 2236
  MedlinePlus_get_genetics_condition_by_name: 2214
  TRIP_Database_Guidelines_Search: 1819
  BiomarkerDiscoveryWorkflow: 1683
  Tool_Finder_LLM: 1658
  FDA_get_indications_by_drug_name: 1333
  EuropePMC_Guidelines_Search: 1245
  MedlinePlus_search_topics_by_keyword: 1242
  euhealthinfo_search_causes_of_death: 1070
  LiteratureSynthesisAgent: 1059
  NICE_Clinical_Guidelines_Search: 1007
  ToolDiscover: 985
  GIN_Guidelines_Search: 948
  FDA_get_mechanism_of_action_by_drug_name: 891
  FDA_get_drug_names_by_indication: 844
  HPA_search_genes_by_query: 769
  euhealthinfo_search_infectious_diseases: 719
  FDA_get_drug_interactions_by_drug_name: 694
  euhealthinfo_search_hospital_in_patient_data: 674
  OpenTargets_get_target_id_descripti

In [29]:
# Get all tool names from ToolUniverse
if isinstance(tu.all_tools, dict):
    available_tools = set(tu.all_tools.keys())
elif isinstance(tu.all_tools, list):
    # Extract tool names from list of dicts
    available_tools = set(t.get('name', t) if isinstance(t, dict) else t for t in tu.all_tools)
else:
    available_tools = set()
    
print(f"Total tools in ToolUniverse: {len(available_tools)}")
print(f"\nSample tools: {list(available_tools)[:10]}")

# Identify meta/agent tools to remove (tools that delegate to other systems)
META_AGENT_TOOLS = {
    # Agent/delegation tools
    'CallAgent',
    'DiseaseAnalyzerAgent', 
    'MedicalLiteratureReviewer',
    'LiteratureSynthesisAgent',
    'ClinicalTrialDesignAgent',
    'DrugSafetyAnalyzer',
    'DrugInteractionAnalyzerAgent',
    'BiomarkerDiscoveryWorkflow',
    
    # Tool finder/discovery tools (model should select directly)
    'Tool_Finder',
    'Tool_Finder_Keyword', 
    'Tool_Finder_LLM',
    'ToolDiscover',
    'Tool_RAG',
    
    # Other meta tools
    'Finish',
}

# Check which meta tools appear in our training data
print("\n=== Meta/Agent Tools in Training Data ===")
for tool in sorted(META_AGENT_TOOLS):
    count = tool_counts.get(tool, 0)
    if count > 0:
        print(f"  {tool}: {count} occurrences")

# Get tools that are in training data but NOT meta tools
concrete_tools_in_data = set(tool_counts.keys()) - META_AGENT_TOOLS
print(f"\n=== Concrete Tools in Training Data: {len(concrete_tools_in_data)} ===")

# Check if these concrete tools exist in ToolUniverse
print("\n=== Validation: Tools in data vs ToolUniverse ===")
in_universe = concrete_tools_in_data & available_tools
not_in_universe = concrete_tools_in_data - available_tools
print(f"Valid (in ToolUniverse): {len(in_universe)}")
print(f"Invalid (NOT in ToolUniverse): {len(not_in_universe)}")
if not_in_universe:
    print(f"\nTools NOT found in ToolUniverse:")
    for t in sorted(not_in_universe)[:30]:
        print(f"  - {t}")

Total tools in ToolUniverse: 768

Sample tools: ['ENCODE_search_experiments', 'kegg_get_pathway_info', 'OpenTargets_get_disease_id_description_by_name', 'get_albumentations_info', 'ToolRelationshipDetector', 'drugbank_get_drug_chemistry_by_drug_name_or_drugbank_id', 'drugbank_get_drug_basic_info_by_drug_name_or_drugbank_id', 'drugbank_full_search', 'OpenTargets_get_drug_approval_status_by_chemblId', 'FDA_get_drug_interactions_by_drug_name']

=== Meta/Agent Tools in Training Data ===
  BiomarkerDiscoveryWorkflow: 1683 occurrences
  CallAgent: 8907 occurrences
  ClinicalTrialDesignAgent: 599 occurrences
  DiseaseAnalyzerAgent: 2918 occurrences
  DrugInteractionAnalyzerAgent: 422 occurrences
  DrugSafetyAnalyzer: 522 occurrences
  LiteratureSynthesisAgent: 1059 occurrences
  MedicalLiteratureReviewer: 2236 occurrences
  ToolDiscover: 985 occurrences
  Tool_Finder: 2641 occurrences
  Tool_Finder_Keyword: 2800 occurrences
  Tool_Finder_LLM: 1658 occurrences

=== Concrete Tools in Training D

In [30]:
# Complete list of tools to REMOVE from training data
TOOLS_TO_REMOVE = META_AGENT_TOOLS | not_in_universe

print(f"=== TOTAL TOOLS TO REMOVE: {len(TOOLS_TO_REMOVE)} ===")
print(f"\n1. Meta/Agent Tools ({len(META_AGENT_TOOLS)}):")
for t in sorted(META_AGENT_TOOLS):
    count = tool_counts.get(t, 0)
    print(f"   - {t}: {count}")

print(f"\n2. Invalid Tools (not in ToolUniverse) ({len(not_in_universe)}):")
for t in sorted(not_in_universe):
    count = tool_counts.get(t, 0)
    print(f"   - {t}: {count}")

# Calculate impact
total_tool_calls_to_remove = sum(tool_counts.get(t, 0) for t in TOOLS_TO_REMOVE)
total_tool_calls = sum(tool_counts.values())
print(f"\n=== IMPACT ===")
print(f"Tool calls to remove: {total_tool_calls_to_remove:,} / {total_tool_calls:,} ({100*total_tool_calls_to_remove/total_tool_calls:.1f}%)")
print(f"Remaining tool calls: {total_tool_calls - total_tool_calls_to_remove:,}")

=== TOTAL TOOLS TO REMOVE: 129 ===

1. Meta/Agent Tools (14):
   - BiomarkerDiscoveryWorkflow: 1683
   - CallAgent: 8907
   - ClinicalTrialDesignAgent: 599
   - DiseaseAnalyzerAgent: 2918
   - DrugInteractionAnalyzerAgent: 422
   - DrugSafetyAnalyzer: 522
   - Finish: 0
   - LiteratureSynthesisAgent: 1059
   - MedicalLiteratureReviewer: 2236
   - ToolDiscover: 985
   - Tool_Finder: 2641
   - Tool_Finder_Keyword: 2800
   - Tool_Finder_LLM: 1658
   - Tool_RAG: 0

2. Invalid Tools (not in ToolUniverse) (115):
   - Analyze: 1
   - Antinuclear Antibody Assay: 1
   - ClinicalStudiesInfo: 4
   - ClinicalStudiesInformationAgent: 1
   - CoxPHFitter: 17
   - DataAnalysis validity and generalizability reviewer: 1
   - Drugbank Filter Drugs by Name: 1
   - Drugbank Get Drug Names by Indication: 2
   - Drugbank Get Drug Names by Name: 1
   - Drugbank Get Drug Names by Pathway: 2
   - FDA Get Drug Names by Indication: 3
   - FDA_get_contraindictions_by_drug_name: 4
   - FDA_get_drug_name_by_abuse_de

In [31]:
# Let's look at a few examples to understand the data format
print("=== EXAMPLE DATA SAMPLES ===\n")

for i, sample in enumerate(all_data[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"INPUT (truncated):\n{sample['input'][:500]}...\n")
    print(f"OUTPUT (all tool calls):\n{sample['output']}\n")
    
    # Count tools in this sample
    tools_in_sample = [line.split(':')[0].strip() for line in sample['output'].split('\n') if ':' in line]
    print(f"Tools called: {len(tools_in_sample)}")
    print("="*60 + "\n")

=== EXAMPLE DATA SAMPLES ===

--- Sample 1 ---
INPUT (truncated):
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle t...

OUTPUT (all tool calls):
FDA_get_indications_by_drug_name: Ampicillin, Ceftriaxone, Ciprofloxacin, Doxycycline, Nitrofurantoin
FDA_get_population_use_info_by_drug_name: Ampicillin, Ceftriaxone, Ciprofloxacin, Doxycycline, Nitrofurantoin

Tools called: 2

--- Sample 2 ---
INPUT (truncated):
Question: A mother brings her 3-week-old infant to the pediatrician's office because she is concerned about his feeding habits. He was bor

In [32]:
# Analyze how many tools per sample
tools_per_sample = []
for sample in all_data:
    tools = [line.split(':')[0].strip() for line in sample['output'].split('\n') if ':' in line and line.split(':')[0].strip()]
    tools_per_sample.append(len(tools))

import numpy as np
print("=== TOOLS PER SAMPLE STATISTICS ===")
print(f"Mean: {np.mean(tools_per_sample):.1f}")
print(f"Median: {np.median(tools_per_sample):.0f}")
print(f"Min: {min(tools_per_sample)}")
print(f"Max: {max(tools_per_sample)}")
print(f"Std: {np.std(tools_per_sample):.1f}")

# Distribution
from collections import Counter
dist = Counter(tools_per_sample)
print(f"\n=== DISTRIBUTION ===")
for n_tools in sorted(dist.keys())[:15]:
    count = dist[n_tools]
    pct = 100 * count / len(tools_per_sample)
    bar = '█' * int(pct/2)
    print(f"{n_tools:2d} tools: {count:4d} samples ({pct:5.1f}%) {bar}")

=== TOOLS PER SAMPLE STATISTICS ===
Mean: 10.3
Median: 4
Min: 1
Max: 226
Std: 13.4

=== DISTRIBUTION ===
 1 tools: 1170 samples ( 16.2%) ████████
 2 tools: 1126 samples ( 15.6%) ███████
 3 tools:  911 samples ( 12.6%) ██████
 4 tools:  592 samples (  8.2%) ████
 5 tools:  332 samples (  4.6%) ██
 6 tools:  225 samples (  3.1%) █
 7 tools:  161 samples (  2.2%) █
 8 tools:  153 samples (  2.1%) █
 9 tools:  159 samples (  2.2%) █
10 tools:  125 samples (  1.7%) 
11 tools:  124 samples (  1.7%) 
12 tools:  123 samples (  1.7%) 
13 tools:  112 samples (  1.6%) 
14 tools:  104 samples (  1.4%) 
15 tools:   86 samples (  1.2%) 


In [37]:
# Clean the dataset: Remove all meta/agent tools and invalid tools

def clean_output(output_text, tools_to_remove):
    """Remove invalid tool lines from output, keeping only valid tools"""
    lines = output_text.strip().split('\n')
    filtered_lines = []
    for line in lines:
        if ':' in line:
            tool_name = line.split(':')[0].strip()
            if tool_name in tools_to_remove:
                continue  # Skip this tool
        filtered_lines.append(line)
    return '\n'.join(filtered_lines).strip()

# Process all data
cleaned_data = []
removed_samples = 0

for sample in all_data:
    output = sample.get('output', '')
    cleaned_output = clean_output(output, TOOLS_TO_REMOVE)
    
    # Only keep samples that still have tool calls after filtering
    if cleaned_output:
        # Verify at least one valid tool remains
        has_valid_tool = False
        for line in cleaned_output.split('\n'):
            if ':' in line:
                tool_name = line.split(':')[0].strip()
                if tool_name and tool_name not in TOOLS_TO_REMOVE:
                    has_valid_tool = True
                    break
        
        if has_valid_tool:
            cleaned_sample = sample.copy()
            cleaned_sample['output'] = cleaned_output
            cleaned_data.append(cleaned_sample)
        else:
            removed_samples += 1
    else:
        removed_samples += 1

print(f"=== CLEANING RESULTS ===")
print(f"Original samples: {len(all_data):,}")
print(f"Removed samples (no valid tools): {removed_samples:,}")
print(f"Cleaned samples: {len(cleaned_data):,}")

# Verify cleaning worked
cleaned_tool_counts = Counter()
for sample in cleaned_data:
    output = sample.get('output', '')
    for line in output.split('\n'):
        if ':' in line:
            tool_name = line.split(':')[0].strip()
            if tool_name:
                cleaned_tool_counts[tool_name] += 1

print(f"\n=== CLEANED DATASET STATS ===")
print(f"Total tool calls: {sum(cleaned_tool_counts.values()):,}")
print(f"Unique tools: {len(cleaned_tool_counts)}")

# Check no invalid tools remain
remaining_invalid = set(cleaned_tool_counts.keys()) & TOOLS_TO_REMOVE
if remaining_invalid:
    print(f"\n⚠️ WARNING: Invalid tools still present: {remaining_invalid}")
else:
    print(f"\n✅ All invalid tools removed!")

print(f"\n=== TOP 50 TOOLS IN CLEANED DATA ===")
for tool, count in cleaned_tool_counts.most_common(50):
    print(f"  {tool}: {count}")

=== CLEANING RESULTS ===
Original samples: 7,212
Removed samples (no valid tools): 808
Cleaned samples: 6,404

=== CLEANED DATASET STATS ===
Total tool calls: 47,564
Unique tools: 487

✅ All invalid tools removed!

=== TOP 50 TOOLS IN CLEANED DATA ===
  get_HPO_ID_by_phenotype: 2359
  MedlinePlus_get_genetics_condition_by_name: 2214
  TRIP_Database_Guidelines_Search: 1819
  FDA_get_indications_by_drug_name: 1333
  EuropePMC_Guidelines_Search: 1245
  MedlinePlus_search_topics_by_keyword: 1242
  euhealthinfo_search_causes_of_death: 1070
  NICE_Clinical_Guidelines_Search: 1007
  GIN_Guidelines_Search: 948
  FDA_get_mechanism_of_action_by_drug_name: 891
  FDA_get_drug_names_by_indication: 844
  HPA_search_genes_by_query: 769
  euhealthinfo_search_infectious_diseases: 719
  FDA_get_drug_interactions_by_drug_name: 694
  euhealthinfo_search_hospital_in_patient_data: 674
  OpenTargets_get_target_id_description_by_name: 662
  euhealthinfo_search_diabetes_mellitus_epidemiology_registry: 646
  Op

In [34]:
# Save cleaned dataset to new files
import os
from sklearn.model_selection import train_test_split

# Create output directory
output_dir = 'datasets/finetuning/cleaned'
os.makedirs(output_dir, exist_ok=True)

# Split into train/val (90/10)
train_data, val_data = train_test_split(cleaned_data, test_size=0.1, random_state=42)

print(f"=== SAVING CLEANED DATASET ===")
print(f"Train samples: {len(train_data):,}")
print(f"Val samples: {len(val_data):,}")

# Save train data
train_file = os.path.join(output_dir, 'model1_tool_selection_train_cleaned.jsonl')
with open(train_file, 'w') as f:
    for sample in train_data:
        f.write(json.dumps(sample) + '\n')
print(f"✅ Saved: {train_file}")

# Save val data
val_file = os.path.join(output_dir, 'model1_tool_selection_val_cleaned.jsonl')
with open(val_file, 'w') as f:
    for sample in val_data:
        f.write(json.dumps(sample) + '\n')
print(f"✅ Saved: {val_file}")

# Also save the full cleaned dataset
full_file = os.path.join(output_dir, 'model1_tool_selection_all_cleaned.jsonl')
with open(full_file, 'w') as f:
    for sample in cleaned_data:
        f.write(json.dumps(sample) + '\n')
print(f"✅ Saved: {full_file}")

# Save list of valid tools for reference
tools_file = os.path.join(output_dir, 'valid_tools.txt')
with open(tools_file, 'w') as f:
    for tool in sorted(cleaned_tool_counts.keys()):
        f.write(f"{tool}\n")
print(f"✅ Saved: {tools_file}")

print(f"\n=== SUMMARY ===")
print(f"Cleaned dataset ready for training!")
print(f"  - {len(train_data):,} train samples")
print(f"  - {len(val_data):,} val samples")
print(f"  - {len(cleaned_tool_counts)} unique valid tools")
print(f"  - {sum(cleaned_tool_counts.values()):,} total tool calls")

=== SAVING CLEANED DATASET ===
Train samples: 5,763
Val samples: 641
✅ Saved: datasets/finetuning/cleaned/model1_tool_selection_train_cleaned.jsonl
✅ Saved: datasets/finetuning/cleaned/model1_tool_selection_val_cleaned.jsonl
✅ Saved: datasets/finetuning/cleaned/model1_tool_selection_all_cleaned.jsonl
✅ Saved: datasets/finetuning/cleaned/valid_tools.txt

=== SUMMARY ===
Cleaned dataset ready for training!
  - 5,763 train samples
  - 641 val samples
  - 487 unique valid tools
  - 47,564 total tool calls


In [36]:
# Analyze cleaned data - tools per sample
cleaned_tools_per_sample = []
for sample in cleaned_data:
    tools = [line.split(':')[0].strip() for line in sample['output'].split('\n') 
             if ':' in line and line.split(':')[0].strip()]
    cleaned_tools_per_sample.append(len(tools))

print("=== CLEANED DATA: TOOLS PER SAMPLE ===")
print(f"Mean: {np.mean(cleaned_tools_per_sample):.1f}")
print(f"Median: {np.median(cleaned_tools_per_sample):.0f}")
print(f"Min: {min(cleaned_tools_per_sample)}")
print(f"Max: {max(cleaned_tools_per_sample)}")

# Distribution
dist = Counter(cleaned_tools_per_sample)
print(f"\n=== DISTRIBUTION ===")
for n_tools in sorted(dist.keys())[:15]:
    count = dist[n_tools]
    pct = 100 * count / len(cleaned_tools_per_sample)
    bar = '█' * int(pct/2)
    print(f"{n_tools:2d} tools: {count:4d} samples ({pct:5.1f}%) {bar}")

# Show a cleaned example
print("\n=== CLEANED SAMPLE EXAMPLE ===")
sample = cleaned_data[0]
print(f"INPUT (first 300 chars):\n{sample['input'][:300]}...\n")
print(f"OUTPUT:\n{sample['output']}")

=== CLEANED DATA: TOOLS PER SAMPLE ===
Mean: 7.4
Median: 3
Min: 1
Max: 142

=== DISTRIBUTION ===
 1 tools: 1586 samples ( 24.8%) ████████████
 2 tools: 1080 samples ( 16.9%) ████████
 3 tools:  764 samples ( 11.9%) █████
 4 tools:  428 samples (  6.7%) ███
 5 tools:  296 samples (  4.6%) ██
 6 tools:  229 samples (  3.6%) █
 7 tools:  174 samples (  2.7%) █
 8 tools:  154 samples (  2.4%) █
 9 tools:  135 samples (  2.1%) █
10 tools:  156 samples (  2.4%) █
11 tools:  118 samples (  1.8%) 
12 tools:  117 samples (  1.8%) 
13 tools:   78 samples (  1.2%) 
14 tools:   90 samples (  1.4%) 
15 tools:   87 samples (  1.4%) 

=== CLEANED SAMPLE EXAMPLE ===
INPUT (first 300 chars):
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature ...

OUTPUT:
FDA